# Generalizable ECG Classification using CNN-Transformer

## Setup

In [ ]:
import numpy as np
import pandas as pd
import wfdb, ast, math
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
from scipy.signal import butter, filtfilt
from scipy.signal import resample as sci_resample
from scipy.ndimage import zoom
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

%matplotlib inline

In [ ]:
# Get data paths and device
LTDB_DIR  = Path('../raw_data/mit-bih-long-term-ecg-database-1.0.0')
PTBXL_DIR = Path('../raw_data/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3')
device    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

## Dataset Understanding

In [ ]:
# LTDB: list records and check one
ltdb_records = [p.stem for p in LTDB_DIR.glob('*.hea') if not p.name.endswith('.hea-')]
print(f'LTDB records ({len(ltdb_records)}):', ltdb_records)

rec = wfdb.rdrecord(str(LTDB_DIR / ltdb_records[0]))
print(f'Fs={rec.fs}Hz | Duration={rec.sig_len/rec.fs/3600:.1f}h | Leads={rec.sig_name}')

# Count all annotation symbols
all_syms = []
for name in ltdb_records:
    ann = wfdb.rdann(str(LTDB_DIR / name), 'atr')
    all_syms.extend(ann.symbol)
print('\nRaw symbol counts:', Counter(all_syms).most_common())

**LTDB:** 7 long recordings , 2 leads, 128Hz, beat-level labels (AAMI standard)  
**PTB-XL:** 21,799 short clips (10s), 12 leads, 100Hz, record-level diagnostic labels  

- Different structure, leads, sampling rates

In [ ]:
# PTB-XL: load metadata and superclass distribution
meta = pd.read_csv(PTBXL_DIR / 'ptbxl_database.csv', index_col='ecg_id')
meta['scp_codes'] = meta['scp_codes'].apply(ast.literal_eval)
scp_df = pd.read_csv(PTBXL_DIR / 'scp_statements.csv', index_col=0)

def get_superclass(scp_dict):
    out = []
    for code, conf in scp_dict.items():
        if conf >= 50 and code in scp_df.index:
            sc = scp_df.loc[code, 'diagnostic_class']
            if pd.notna(sc): out.append(sc)
    return list(set(out))

meta['superclass'] = meta['scp_codes'].apply(get_superclass)
print('PTB-XL superclass counts:')
print(meta['superclass'].explode().value_counts())

## Preprocessing & Feature Extraction
**Bandpass filter (0.5–40Hz):** Removes baseline wander (<0.5Hz from breathing) and high-freq noise (>40Hz from muscle/electrical). 

**Z-score normalization:** Different equipment records at different voltage scales. Normalization makes the model focus on *shape*, not *amplitude*.  

**Beat segmentation (LTDB):** R-peak is the reference point of every heartbeat. Window of 100 samples before + 150 after captures the full P-QRS-T complex.  clinicians examine this to classify arrhythmias.  

- P-wave presence/absence identifies supraventricular (S). 
- QRS width identifies ventricular beats (V). 
- T-wave shape identifies fusion (F).

In [ ]:
def bandpass_filter(sig, fs, lo=0.5, hi=40.0, order=4):
    nyq = fs / 2
    b, a = butter(order, [lo/nyq, hi/nyq], btype='band')
    return filtfilt(b, a, sig, axis=0)

def normalize(sig):
    return (sig - sig.mean(axis=0)) / (sig.std(axis=0) + 1e-8)

# Visualize filter effect
rec = wfdb.rdrecord(str(LTDB_DIR / ltdb_records[0]))
sig_raw  = rec.p_signal
sig_filt = bandpass_filter(sig_raw, rec.fs)
t = 5 * rec.fs

fig, axes = plt.subplots(2, 1, figsize=(12, 4), sharex=True)
for i, lead in enumerate(rec.sig_name[:2]):
    axes[i].plot(sig_raw[:t, i],  lw=0.7, color='gray',      label='raw')
    axes[i].plot(sig_filt[:t, i], lw=0.8, color='steelblue', label='filtered')
    axes[i].set_ylabel(f'{lead} (mV)'); axes[i].legend(fontsize=8)
axes[1].set_xlabel('Samples')
fig.suptitle('Bandpass filter effect') # Removes drift and noise, preserves P-QRS-T
plt.tight_layout(); plt.show()

In [ ]:
# LTDB Beat Extraction
AAMI_MAP   = {'N':'N','L':'N','R':'N','e':'N','j':'N',
              'A':'S','a':'S','J':'S','S':'S',
              'V':'V','E':'V','F':'F'}
AAMI_LABEL = {'N':0, 'S':1, 'V':2, 'F':3}
WIN_PRE, WIN_POST = 100, 150

def extract_beats(record_name):
    rec = wfdb.rdrecord(str(LTDB_DIR / record_name))
    ann = wfdb.rdann(str(LTDB_DIR / record_name), 'atr')
    sig = rec.p_signal.astype(np.float32)[:, :2]
    sig = bandpass_filter(sig, rec.fs)
    sig = normalize(sig)
    beats, labels = [], []
    for r, sym in zip(ann.sample, ann.symbol):
        if sym not in AAMI_MAP: continue
        if r < WIN_PRE or r + WIN_POST > len(sig): continue
        beats.append(sig[r-WIN_PRE:r+WIN_POST].T)
        labels.append(AAMI_LABEL[AAMI_MAP[sym]])
    return np.stack(beats), np.array(labels)

ltdb_X, ltdb_y = [], []
for name in ltdb_records:
    X, y = extract_beats(name)
    ltdb_X.append(X); ltdb_y.append(y)
    print(f'  {name}: {X.shape[0]:,} beats | {Counter(y)}')

ltdb_X = np.concatenate(ltdb_X)
ltdb_y = np.concatenate(ltdb_y)
print(f'\nLTDB total: {ltdb_X.shape} | {Counter(ltdb_y)}')

In [ ]:
# PTB-XL Extraction
SC_MAP = {'NORM':0, 'MI':1, 'STTC':2, 'CD':3, 'HYP':4}

def load_ptbxl(row):
    rec = wfdb.rdrecord(str(PTBXL_DIR / row.filename_lr))
    sig = rec.p_signal.astype(np.float32)
    sig = bandpass_filter(sig, 100)
    sig = normalize(sig)
    return sig.T  # (12, 1000)

ptbxl_X, ptbxl_y = [], []
for ecg_id, row in meta.iterrows():
    scs = row['superclass']
    if len(scs) != 1: continue
    label = SC_MAP.get(scs[0], -1)
    if label < 0: continue
    ptbxl_X.append(load_ptbxl(row))
    ptbxl_y.append(label)

ptbxl_X = np.stack(ptbxl_X)
ptbxl_y = np.array(ptbxl_y)
print(f'PTB-XL total: {ptbxl_X.shape} | {Counter(ptbxl_y)}')

LTDB: preprocessed beats per class
- V=wide QRS, 
- N=narrow sharp peak
- F=hybrid shape

In [ ]:
# Visualization: one beat per class
fig, axes = plt.subplots(4, 1, figsize=(11, 7), sharex=True)
names  = {0:'N — Normal', 1:'S — Supraventricular', 2:'V — Ventricular', 3:'F — Fusion'}
colors = {0:'steelblue', 1:'darkorange', 2:'crimson', 3:'green'}

for cls in range(4):
    idx = np.where(ltdb_y == cls)[0][0]
    axes[cls].plot(ltdb_X[idx, 0], lw=0.9, color=colors[cls])
    axes[cls].axvline(100, color='gray', lw=0.6, ls='--', label='R-peak')
    axes[cls].set_ylabel(names[cls], fontsize=9)
    axes[cls].legend(fontsize=7)

axes[-1].set_xlabel('Samples  — R-peak at sample 100')

plt.tight_layout(); plt.show()

# Class imbalance
fig, axes = plt.subplots(1, 2, figsize=(11, 3))
pd.Series(ltdb_y).value_counts().sort_index().plot.bar(
    ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_xticklabels(['N','S','V','F'], rotation=0)
axes[0].set_title('LTDB class distribution')
axes[0].set_yscale('log')

pd.Series(ptbxl_y).value_counts().sort_index().plot.bar(
    ax=axes[1], color='coral', edgecolor='white')
axes[1].set_xticklabels(['NORM','MI','STTC','CD','HYP'], rotation=0)
axes[1].set_title('PTB-XL class distribution')
plt.tight_layout(); plt.show()

## Model: CNN-Transformer

- CNN first because CNNs scan with small filters. They learn local morphological patterns (QRS shape, P-wave, T-wave). This is what a doctor looks at in a single beat.  
-  Transformer after as it captures relationships across the whole sequence. The CLS token summarizes the entire signal into one vector for classification.  

**Residual connections:** Allow gradients to flow during training without vanishing which makes deeper networks trainable.  
**Two separate models:** One per dataset (different number of leads and classes).

In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, k):
        super().__init__()
        self.net  = nn.Sequential(
            nn.Conv1d(in_ch, out_ch, k, padding=k//2, bias=False),
            nn.BatchNorm1d(out_ch), nn.GELU(),
            nn.Conv1d(out_ch, out_ch, k, padding=k//2, bias=False),
            nn.BatchNorm1d(out_ch),
        )
        self.skip = nn.Conv1d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()
        self.act  = nn.GELU()
    def forward(self, x):
        return self.act(self.net(x) + self.skip(x))


class ECGNet(nn.Module):
    """
    CNN-Transformer for ECG classification.
    CNN: extracts local beat morphology (QRS, P-wave, T-wave shapes).
    Transformer: models temporal context and global relationships.
    CLS token: aggregates full sequence into one classification vector.
    """
    def __init__(self, in_ch, n_classes, d_model=128, n_heads=4, n_layers=3):
        super().__init__()
        self.cnn = nn.Sequential(
            ConvBlock(in_ch, 32,      k=7),  # broad: P and T waves
            nn.MaxPool1d(2),
            ConvBlock(32,   64,      k=5),  # medium: QRS complex
            nn.MaxPool1d(2),
            ConvBlock(64,   d_model, k=3),  # fine: sharp peaks
        )
        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_model))
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        enc = nn.TransformerEncoderLayer(
            d_model, n_heads, dim_feedforward=256, dropout=0.1,
            activation='gelu', batch_first=True, norm_first=True)
        self.transformer = nn.TransformerEncoder(enc, n_layers, norm=nn.LayerNorm(d_model))
        self.head = nn.Linear(d_model, n_classes)

    def forward(self, x):
        z   = self.cnn(x).permute(0, 2, 1)
        cls = self.cls_token.expand(x.size(0), -1, -1)
        z   = torch.cat([cls, z], dim=1)
        z   = self.transformer(z)
        return self.head(z[:, 0])

    def get_attn(self, x):
        self.eval()
        z = self.cnn(x).permute(0, 2, 1)
        cls = self.cls_token.expand(x.size(0), -1, -1)
        z   = torch.cat([cls, z], dim=1)
        with torch.no_grad():
            for layer in self.transformer.layers[:-1]:
                z = layer(z)
            _, attn = self.transformer.layers[-1].self_attn(
                z, z, z, need_weights=True, average_attn_weights=True)
        return attn[0, 0, 1:].cpu().numpy()

model_ltdb  = ECGNet(in_ch=2,  n_classes=4).to(device)
model_ptbxl = ECGNet(in_ch=12, n_classes=5).to(device)

# Parameter count
def count_params(m): return sum(p.numel() for p in m.parameters() if p.requires_grad)
print(f'LTDB  model params: {count_params(model_ltdb):,}')
print(f'PTB-XL model params: {count_params(model_ptbxl):,}')

## Training Strategy
**Supervised learning with class-imbalance correction**

Reason: We have labels for all data. Semi-supervised would be beneficial if labels were scarce, but here we have 600k+ labeled beats.

**Key decisions:**
1. **WeightedRandomSampler:** Weighted sampling ensures balanced class exposure.LTDB is 90% N-beats.So, without this, the model predicts N always and achieves 90% accuracy while being clinically useless. 
2. **Label smoothing (0.1):** Prevents overconfident predictions and improves calibration and generalization.
3. **AdamW:** Better weight decay handling than Adam and prevents overfitting on dominant classes.
4. **CosineAnnealingLR:** Smooth learning rate decay and avoids abrupt drops that destabilize training.
5. **Data augmentation:** Gaussian noise & amplitude jitter creates slightly varied versions of each beat. Model learns morphology, not exact signal values.
6. **Gradient clipping (1.0):** Prevents exploding gradients in the Transformer layers.

In [ ]:
class ECGDataset(Dataset):
    def __init__(self, X, y, augment=False):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
        self.augment = augment
    def __len__(self): return len(self.y)
    def __getitem__(self, i):
        x = self.X[i].clone()
        if self.augment:
            x += torch.randn_like(x) * 0.05
            x *= float(np.random.uniform(0.9, 1.1))
        return x, self.y[i]

def make_loaders(X, y, batch=64):
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.15, stratify=y, random_state=42)
    Xtr, Xvl, ytr, yvl = train_test_split(Xtr, ytr, test_size=0.15, stratify=ytr, random_state=42)
    w  = 1.0 / np.bincount(ytr)[ytr]
    tr = DataLoader(ECGDataset(Xtr, ytr, augment=True), batch, sampler=WeightedRandomSampler(w, len(w)))
    vl = DataLoader(ECGDataset(Xvl, yvl), batch, shuffle=False)
    te = DataLoader(ECGDataset(Xte, yte),  batch, shuffle=False)
    return tr, vl, te, Xte, yte

def train(model, tr, vl, epochs=40, lr=3e-4, tag=''):
    opt     = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched   = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs)
    loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1)
    best_f1, best_w = 0, None
    history = []
    for ep in range(epochs):
        model.train()
        for x, y in tr:
            x, y = x.to(device), y.to(device)
            loss = loss_fn(model(x), y)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); opt.zero_grad()
        sched.step()
        model.eval()
        ps, ts = [], []
        with torch.no_grad():
            for x, y in vl:
                ps.append(model(x.to(device)).argmax(1).cpu()); ts.append(y)
        f1 = f1_score(torch.cat(ts), torch.cat(ps), average='macro')
        history.append(f1)
        if f1 > best_f1:
            best_f1 = f1
            best_w  = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        if (ep+1) % 10 == 0:
            print(f'  [{tag}] ep {ep+1:02d} | val F1={f1:.3f}')
    model.load_state_dict(best_w)
    print(f'  [{tag}] Best val F1: {best_f1:.3f}')
    return history

tr_l, vl_l, te_l, Xte_l, yte_l = make_loaders(ltdb_X,  ltdb_y)
tr_p, vl_p, te_p, Xte_p, yte_p = make_loaders(ptbxl_X, ptbxl_y)

print('=== Training LTDB ===')
hist_l = train(model_ltdb,  tr_l, vl_l, tag='LTDB')
print('\n=== Training PTB-XL ===')
hist_p = train(model_ptbxl, tr_p, vl_p, tag='PTB-XL')

# Training curves
fig, axes = plt.subplots(1, 2, figsize=(11, 3))
for ax, hist, title in zip(axes, [hist_l, hist_p], ['LTDB', 'PTB-XL']):
    ax.plot(hist, color='steelblue'); ax.set_xlabel('Epoch'); ax.set_ylabel('Val Macro F1')
    ax.set_title(f'{title} training curve'); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## Evaluation

In [ ]:
def evaluate(model, loader, class_names, title):
    model.eval()
    ps, ts = [], []
    with torch.no_grad():
        for x, y in loader:
            ps.append(model(x.to(device)).argmax(1).cpu()); ts.append(y)
    y_pred = torch.cat(ps).numpy()
    y_true = torch.cat(ts).numpy()
    print(f'\n= {title} =')
    print(classification_report(y_true, y_pred, target_names=class_names))
    cm = confusion_matrix(y_true, y_pred, normalize='true')
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='.2f', cmap='Blues',xticklabels=class_names, yticklabels=class_names)
    plt.title(title); plt.xlabel('Predicted'); plt.ylabel('True')
    plt.tight_layout(); plt.show()
    return y_true, y_pred

true_l, pred_l = evaluate(model_ltdb,  te_l, ['N','S','V','F'],           'LTDB Test')
true_p, pred_p = evaluate(model_ptbxl, te_p, ['NORM','MI','STTC','CD','HYP'], 'PTB-XL Test')

## Interpretability

A model can be accurate for wrong reasons. If it classifies V-beats based on noise artifacts rather than QRS morphology, it will fail on clean data from another hospital.

**Attention map:** Shows which time steps the Transformer focused on. If peaks align with the QRS region (around sample 100), the model is reasoning about beat morphology which is clinically correct.

**Grad-CAM:** Computes gradient of the predicted class score with respect to the input signal. High gradient means changing that sample would most change the prediction. It reveals which signal regions drive classification decisions.

In [ ]:
# Attention maps
fig, axes = plt.subplots(4, 1, figsize=(12, 9), sharex=False)
for cls in range(4):
    idx    = np.where(ltdb_y == cls)[0][0]
    sample = ltdb_X[idx]
    x      = torch.tensor(sample[np.newaxis], dtype=torch.float32).to(device)
    attn   = model_ltdb.get_attn(x)
    attn_up = zoom(attn, sample.shape[1] / len(attn))
    ax  = axes[cls]; ax2 = ax.twinx()
    ax.plot(sample[0], lw=0.8, color=list(colors.values())[cls], label='signal')
    ax2.fill_between(np.arange(250), attn_up, alpha=0.4, color='gold', label='attention')
    ax.axvline(100, color='gray', lw=0.5, ls='--')
    ax.set_ylabel(list(names.values())[cls], fontsize=9)
    ax.legend(fontsize=7, loc='upper left'); ax2.set_ylabel('attn', fontsize=7)
axes[-1].set_xlabel('Samples')
plt.tight_layout(); plt.show()

- Attention map: Gold = where transformer focuses
- Meaningful if peaks near QRS

In [ ]:
# Grad-CAM
fig, axes = plt.subplots(4, 1, figsize=(12, 9), sharex=True)
for cls in range(4):
    idx    = np.where(ltdb_y == cls)[0][0]
    sample = ltdb_X[idx]
    x = torch.tensor(sample[np.newaxis], dtype=torch.float32, requires_grad=True).to(device)
    model_ltdb.eval()
    logits = model_ltdb(x)
    model_ltdb.zero_grad()
    logits[0, cls].backward()
    cam = np.abs(x.grad.detach().cpu().numpy()[0]).mean(axis=0)
    cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
    axes[cls].plot(sample[0], lw=0.8, color=list(colors.values())[cls])
    axes[cls].fill_between(np.arange(250), sample[0].min() * cam, sample[0].max() * cam, alpha=0.4, color='gold')
    axes[cls].axvline(100, color='gray', lw=0.5, ls='--')
    axes[cls].set_ylabel(list(names.values())[cls], fontsize=9)
axes[-1].set_xlabel('Samples')
plt.tight_layout(); plt.show()

- Grad-CAM: Gold = most influential signal regions
- V beats should highlight wide QRS, N beats the sharp peak

## Generalizability

1. **Held-out test split (15%):** Same dataset, unseen patients.
2. **Gaussian noise test:** Adds increasing noise to test signals. Real-world ECGs from different equipment have different noise levels.
3. **OOD test (MIT-BIH):** Completely different database — different patients, hospital, equipment, 360Hz sampling rate.

In [ ]:
# Noise robustness
noise_levels = [0.0, 0.05, 0.1, 0.2, 0.5]
f1_noise = []
for std in noise_levels:
    Xn = Xte_l + np.random.randn(*Xte_l.shape).astype(np.float32) * std
    loader = DataLoader(ECGDataset(Xn, yte_l), batch_size=64, shuffle=False)
    ps, ts = [], []
    model_ltdb.eval()
    with torch.no_grad():
        for x, y in loader:
            ps.append(model_ltdb(x.to(device)).argmax(1).cpu()); ts.append(y)
    f1 = f1_score(torch.cat(ts), torch.cat(ps), average='macro')
    f1_noise.append(f1)
    print(f'  Noise sigma={std:.2f} -> Macro F1={f1:.3f}')

plt.figure(figsize=(7, 4))
plt.plot(noise_levels, f1_noise, 'o-', color='steelblue')
plt.xlabel('Gaussian noise sigma'); plt.ylabel('Macro F1')
plt.title('Noise robustness')
plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

- Flat curve = robust
- Sharp drop = fragile/overfit.

In [ ]:
# OOD test: MIT-BIH Arrhythmia Database
MITDB_DIR = Path('../raw_data/mit-bih-arrhythmia-database-1.0.0')
if not MITDB_DIR.exists():
    MITDB_DIR.mkdir(parents=True)
    wfdb.dl_database('mitdb', str(MITDB_DIR))

mitdb_records = [p.stem for p in MITDB_DIR.glob('*.hea') if not p.name.endswith('.hea-')]
print(f'MIT-BIH records: {len(mitdb_records)}')

def extract_beats_mitdb(name):
    rec = wfdb.rdrecord(str(MITDB_DIR / name))
    ann = wfdb.rdann(str(MITDB_DIR / name), 'atr')
    sig = rec.p_signal.astype(np.float32)[:, :2]
    n   = int(sig.shape[0] * 128 / rec.fs)
    sig = sci_resample(sig, n, axis=0)
    r_peaks = (ann.sample * 128 / rec.fs).astype(int)
    sig = bandpass_filter(sig, 128)
    sig = normalize(sig)
    beats, labels = [], []
    for r, sym in zip(r_peaks, ann.symbol):
        if sym not in AAMI_MAP: continue
        if r < WIN_PRE or r + WIN_POST > len(sig): continue
        beats.append(sig[r-WIN_PRE:r+WIN_POST].T)
        labels.append(AAMI_LABEL[AAMI_MAP[sym]])
    return np.stack(beats), np.array(labels)

ood_X, ood_y = [], []
for name in mitdb_records:
    try:
        X, y = extract_beats_mitdb(name)
        ood_X.append(X); ood_y.append(y)
    except: pass

ood_X = np.concatenate(ood_X)
ood_y = np.concatenate(ood_y)
print(f'OOD beats: {ood_X.shape} | {Counter(ood_y)}')

ood_loader = DataLoader(ECGDataset(ood_X, ood_y), batch_size=64, shuffle=False)
true_ood, pred_ood = evaluate(model_ltdb, ood_loader, ['N','S','V','F'],
                               'OOD Test: MIT-BIH (never seen during training)')

in_f1  = f1_score(true_l,   pred_l,   average='macro')
ood_f1 = f1_score(true_ood, pred_ood, average='macro')
drop   = (in_f1 - ood_f1) / in_f1 * 100

print(f'\nIn-distribution F1  (LTDB test) : {in_f1:.3f}')
print(f'Out-of-distribution F1 (MIT-BIH): {ood_f1:.3f}')
print(f'Relative drop                    : {drop:.1f}%')
if   drop < 15: print('-> DROP <15%: Model generalizes well.')
elif drop < 30: print('-> DROP 15-30%: Moderate generalization.')
else:           print('-> DROP >30%: Model is dataset-specific.')